# Part 1.2 — The same agent, with LangGraph

In the last notebook you built an agent as a raw loop. Now we build the
**same agent** with [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview)
— the framework NVIDIA's original workshop uses too.

The point of doing it twice: you already know what the loop is, so you can
see exactly what the framework adds — and decide for yourself when it
earns its place.

> **Before you start:** [`../setup.md`](../setup.md) done, `python ../llm.py`
> prints `connection ok`.

In [ ]:
import sys

sys.path.append("..")
from llm import chat_model

from langchain.agents import create_agent
from langchain_core.tools import tool

import tools as commerce  # the same search_products / get_product_details

## Step 1 — Wrap the tools

In notebook 01 you hand-wrote a JSON schema for each tool. The `@tool`
decorator does that for you: it reads the function signature and the
docstring and builds the schema automatically.

Same two functions as before — just decorated.

In [ ]:
@tool
def search_products(query: str) -> str:
    """Search the product catalogue by keyword. Returns matching products as JSON."""
    return commerce.search_products(query)


@tool
def get_product_details(product_id: str) -> str:
    """Get full details for one product by its product_id (e.g. 'JKT-001'). Returns JSON."""
    return commerce.get_product_details(product_id)


toolbox = [search_products, get_product_details]
print("Tools ready:", [t.name for t in toolbox])

## Step 2 — Create the agent

`create_agent` replaces your entire `run_agent` loop. It builds the
model -> tool -> model cycle as a graph and runs it for you.

In [ ]:
SYSTEM_PROMPT = """You are a shopping assistant for an outdoor-gear shop.

- Use search_products to find products, and get_product_details when you
  need more detail about a specific item.
- Search at most twice, and never repeat a search you have already run.
- As soon as you have enough information, STOP calling tools and write
  your answer.
- Recommend at most three products. For each, give the price in CHF and
  whether it is in stock.
- Never invent products, prices or stock — state only what the tools
  returned. If nothing matches the request well, say so honestly and
  suggest the closest alternative."""

agent = create_agent(
    model=chat_model(temperature=0.1),
    tools=toolbox,
    system_prompt=SYSTEM_PROMPT,
)
print("Agent created.")

## Step 3 — Run it

The agent takes a list of messages and returns the updated list. The last
message is the answer.

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I need a warm jacket for winter, budget around CHF 250.",
            }
        ]
    }
)
print(result["messages"][-1].content)

## The loop is still there

The framework did not remove the loop — it just runs it for you. Print
every message and you see the same model -> tool -> model cycle from
notebook 01.

In [ ]:
for message in result["messages"]:
    message.pretty_print()

## What the framework adds — memory

The raw loop forgot everything between questions. LangGraph gives the
agent memory with a **checkpointer**: pass one in, call the agent with a
`thread_id`, and the conversation persists across calls.

Watch the second question — it says "those", with no other context.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=chat_model(temperature=0.1),
    tools=toolbox,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

conversation = {"configurable": {"thread_id": "shopper-1"}}

first = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Show me your hiking boots."}]},
    conversation,
)
print("Q1: Show me your hiking boots.")
print("A1:", first["messages"][-1].content)
print()

second = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Which of those is waterproof?"}]},
    conversation,
)
print("Q2: Which of those is waterproof?")
print("A2:", second["messages"][-1].content)

## Raw loop vs. LangGraph — when to use which

| | Raw loop (notebook 01) | LangGraph (this notebook) |
|---|---|---|
| Lines of agent code | ~40, all yours | ~3 |
| Tool schemas | Hand-written | Generated from `@tool` |
| Memory across turns | You build it | `checkpointer` |
| Streaming, retries, tracing | You build it | Built in |
| Every moving part visible | Yes | Mostly hidden |

Neither is "better". Build the raw loop when you are **learning**, or need
total control over an unusual flow. Reach for LangGraph when you are
**shipping** and want memory, streaming and tracing without re-inventing
them.

---

**From demo to production.** Both agents still answer from a 12-product
JSON file with keyword search. **Part 2** replaces that with *retrieval* —
embeddings over a real catalogue — so the agent finds the right product
even when the customer's words do not match the product's words.